# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. `20 10`. `s["b"]` берёт значение по ярлыку `b`, а `s.iloc[0]` берёт первое значение по позиции.
2. Астана — `NaN`, Алматы — `NaN`, Шымкент — `3.0`. Pandas складывает значения с одинаковыми ярлыками. Только Шымкент есть в обеих Series.
3. `(2, 2)`. Фильтр оставляет две строки, где `x` равно 2 или 3, и оба столбца.
4. `Series DataFrame`. `df["x"]` возвращает один столбец, а `df[["x"]]` выбирает таблицу из одного столбца.
5. `[1, 2] 5`. После фильтра остаются исходные ярлыки 1 и 2. `iloc[0, 1]` берёт `y` в первой из оставшихся строк.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [4]:
# 1
print(f"Строк: {df.shape[0]}, столбцов: {df.shape[1]}")
print(df.dtypes)

missing_days = len(df) - df["delivery_days"].count()
print(f"Пропусков в delivery_days: {missing_days}")
print("Разных значений:")
print("city:", df["city"].nunique())
print("category:", df["category"].nunique())
print("status:", df["status"].nunique())
print("customer_id:", df["customer_id"].nunique())

test_count = (~not_test).sum()
extra_rows = len(df) - df["order_id"].nunique()
print(f"У клиента TEST {test_count} заказа. Исключите их, иначе они завысят выручку.")
print(f"Есть {extra_rows} лишних строк с повторяющимися order_id. Иначе оплаченный заказ может попасть в сумму дважды.")
print("У городов разные написания. Объедините варианты, иначе выручка одного города разойдётся по разным строкам отчёта.")

Строк: 5463, столбцов: 8
order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object
Пропусков в delivery_days: 160
Разных значений:
city: 14
category: 5
status: 2
customer_id: 2003
У клиента TEST 3 заказа. Исключите их, иначе они завысят выручку.
Есть 180 лишних строк с повторяющимися order_id. Иначе оплаченный заказ может попасть в сумму дважды.
У городов разные написания. Объедините варианты, иначе выручка одного города разойдётся по разным строкам отчёта.


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [5]:
# 2
checks = df.loc[paid & not_test, "amount"]
average = checks.mean()
median = checks.median()
q1 = checks.quantile(0.25)
q3 = checks.quantile(0.75)

print(f"Среднее: {average:,.2f} ₸")
print(f"Медиана: {median:,.0f} ₸")
print(f"25-й перцентиль: {q1:,.0f} ₸, 75-й перцентиль: {q3:,.0f} ₸")
print(f"Для типичного заказа покажите медиану {median:,.0f} ₸. Дорогие заказы поднимают среднее до {average:,.2f} ₸.")

Среднее: 14,672.39 ₸
Медиана: 10,300 ₸
25-й перцентиль: 5,880 ₸, 75-й перцентиль: 18,690 ₸
Для типичного заказа покажите медиану 10,300 ₸. Дорогие заказы поднимают среднее до 14,672.39 ₸.


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [6]:
# 3
# Варианты написания из df["city"].value_counts().
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
shymkent = df["city"].isin(SH)
almaty = df["city"].isin(AL)
cancelled = df["status"] == "отменён"

all_cancel = cancelled.mean()
sh_cancel = cancelled[shymkent].mean()
al_cancel = cancelled[almaty].mean()

print(f"Все заказы: {all_cancel:.1%}, Шымкент: {sh_cancel:.1%}, Алматы: {al_cancel:.1%}")
print(f"В Шымкенте доля отмен в {sh_cancel / al_cancel:.1f} раза выше, чем в Алматы, "
      f"и в {sh_cancel / all_cancel:.1f} раза выше общей доли. Начните проверку причин отмен с Шымкента.")

Все заказы: 10.2%, Шымкент: 20.4%, Алматы: 7.9%
В Шымкенте доля отмен в 2.6 раза выше, чем в Алматы, и в 2.0 раза выше общей доли. Начните проверку причин отмен с Шымкента.


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [7]:
# 4
shymkent = df["city"].isin(SH)
almaty = df["city"].isin(["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"])
include = paid & not_test

sh_sep = df.loc[include & shymkent & sep, "amount"].sum()
sh_oct = df.loc[include & shymkent & octo, "amount"].sum()
al_sep = df.loc[include & almaty & sep, "amount"].sum()
al_oct = df.loc[include & almaty & octo, "amount"].sum()
sh_change = (sh_oct - sh_sep) / sh_sep
al_change = (al_oct - al_sep) / al_sep

print(f"Шымкент: {sh_sep:,.0f} ₸ в сентябре, {sh_oct:,.0f} ₸ в октябре, изменение {sh_change:+.1%}")
print(f"Алматы: {al_sep:,.0f} ₸ в сентябре, {al_oct:,.0f} ₸ в октябре, изменение {al_change:+.1%}")
print(f"Падение выручки в процентах в Шымкенте в {sh_change / al_change:.1f} раза сильнее. "
      "Сначала проверьте повторы заказов и причины отмен в Шымкенте.")

Шымкент: 4,949,800 ₸ в сентябре, 4,044,010 ₸ в октябре, изменение -18.3%
Алматы: 15,523,940 ₸ в сентябре, 14,643,020 ₸ в октябре, изменение -5.7%
Падение выручки в процентах в Шымкенте в 3.2 раза сильнее. Сначала проверьте повторы заказов и причины отмен в Шымкенте.


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [8]:
# 5
# 1. & выполняется раньше ==, поэтому каждое условие нужно взять в скобки.
# 2. Сравнение с одним написанием пропускает «шымкент» и «Shymkent».
cancel_one_name = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]
cancel_sh = df[df["city"].isin(SH) & (df["status"] == "отменён")]

print(f"С одним написанием: {len(cancel_one_name)}. Со всеми написаниями: {len(cancel_sh)}.")
print(f"Для отчёта берите все {len(cancel_sh)} отмены. Иначе потеряются {len(cancel_sh) - len(cancel_one_name)} строк.")

С одним написанием: 153. Со всеми написаниями: 164.
Для отчёта берите все 164 отмены. Иначе потеряются 11 строк.


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [9]:
# 6
customer_counts = df["customer_id"].value_counts()
top_customer = customer_counts.index[0]
top_mask = df["customer_id"] == top_customer
top_orders = df.loc[top_mask, ["order_date", "city", "amount", "status"]]

print(f"Больше всего строк у клиента {top_customer}: {len(top_orders)}")
print(top_orders)

top_ids = df.loc[top_mask, "order_id"]
id_counts = top_ids.value_counts()
repeated_ids = id_counts[id_counts > 1]
repeated_id = repeated_ids.index[0]
repeat_amount = df.loc[top_mask & paid & (df["order_id"] == repeated_id), "amount"].iloc[0]

print(f"Уникальных заказов: {top_ids.nunique()}")
print(f"Заказ {repeated_id} встречается {repeated_ids.iloc[0]} раза.")
print(f"В выручку этого клиента лишний раз попало {repeat_amount:,.0f} ₸. "
      "Перед итоговым отчётом проверьте повторяющиеся order_id во всей таблице.")

Больше всего строк у клиента C01818: 11
            order_date       city   amount   status
57    2026-09-01 15:00    Шымкент   7350.0  оплачен
779   2026-09-09 18:00    Шымкент  45270.0  оплачен
1371  2026-09-15 21:00     Алматы   8890.0  оплачен
1375  2026-09-15 21:00     Алматы   8890.0  оплачен
2195  2026-09-24 18:00     Астана   1490.0  оплачен
2617  2026-09-29 09:00     Астана  22170.0  оплачен
2776  2026-09-30 21:00     Алматы  20180.0  оплачен
3417  2026-10-08 14:00  Караганда   4330.0  оплачен
4279  2026-10-18 08:00     Астана  13600.0  оплачен
4610  2026-10-21 19:00     Алматы  21990.0  оплачен
4778  2026-10-23 18:00     Алматы  11140.0  оплачен
Уникальных заказов: 10
Заказ 101504 встречается 2 раза.
В выручку этого клиента лишний раз попало 8,890 ₸. Перед итоговым отчётом проверьте повторяющиеся order_id во всей таблице.


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [10]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [11]:
# 8